In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

In [2]:
files = [
    "../data/CRMLSSold202512.csv",
    "../data/CRMLSSold202601.csv",
    "../data/CRMLSSold202602.csv",
    "../data/CRMLSSold202603.csv",
    "../data/CRMLSSold202604.csv",
    "../data/CRMLSSold202605.csv"
]

dfs = [pd.read_csv(file) for file in files]

df = pd.concat(dfs, ignore_index=True)

print(df.shape)

C:\Users\sonng\AppData\Local\Temp\ipykernel_25404\2146585171.py:10: DtypeWarning: Columns (4,74) have mixed types. Specify dtype option on import or set low_memory=False.
  dfs = [pd.read_csv(file) for file in files]


(124404, 78)


In [3]:
df = df[
    (df["PropertyType"] == "Residential") &
    (df["PropertySubType"] == "SingleFamilyResidence")
]

print(df.shape)

(61727, 78)


In [4]:
features = [
    "ClosePrice",
    "LivingArea",
    "BedroomsTotal",
    "BathroomsTotalInteger",
    "LotSizeSquareFeet",
    "YearBuilt",
    "City",
    "CountyOrParish",
    "ListPrice",
    "CloseDate"
]

df = df[features]

df.head()

,ClosePrice,LivingArea,BedroomsTotal,BathroomsTotalInteger,LotSizeSquareFeet,YearBuilt,City,CountyOrParish,ListPrice,CloseDate
0,1998000.0,2045.0,4.0,2.0,10080.0,1968.0,Walnut Creek,Contra Costa,1998000.0,2025-12-31
2,2214421.0,3050.0,4.0,4.0,34745.0,1957.0,Woodland Hills,Los Angeles,2214421.0,2025-12-31
3,1200000.0,1594.0,4.0,2.0,6600.0,1978.0,San Jose,Santa Clara,1200000.0,2025-12-31
7,3100000.0,2700.0,5.0,3.0,8262.0,2025.0,San Jose,Santa Clara,3100000.0,2025-12-31
9,2900000.0,2948.0,5.0,4.0,9222.0,2023.0,San Jose,Santa Clara,2900000.0,2025-12-31


In [5]:
missing_pct = (
    df.isnull()
      .mean()
      .sort_values(ascending=False)
      * 100
)

missing_pct

LotSizeSquareFeet        1.751260
YearBuilt                0.058321
LivingArea               0.048601
City                     0.022681
BathroomsTotalInteger    0.001620
ClosePrice               0.000000
BedroomsTotal            0.000000
CountyOrParish           0.000000
ListPrice                0.000000
CloseDate                0.000000
dtype: float64

In [6]:
threshold = 70

keep_cols = missing_pct[missing_pct < threshold].index

df = df[keep_cols]

print(df.columns)

Index(['LotSizeSquareFeet', 'YearBuilt', 'LivingArea', 'City',
       'BathroomsTotalInteger', 'ClosePrice', 'BedroomsTotal',
       'CountyOrParish', 'ListPrice', 'CloseDate'],
      dtype='object')


In [7]:
df["CloseDate"] = pd.to_datetime(df["CloseDate"])

In [8]:
df["PropertyAge"] = (
    df["CloseDate"].dt.year
    - df["YearBuilt"]
)

In [9]:
df = df.drop(columns=["YearBuilt"])

In [10]:
numerical_cols = [
    "LivingArea",
    "BedroomsTotal",
    "BathroomsTotalInteger",
    "LotSizeSquareFeet",
    "ListPrice",
    "PropertyAge"
]

for col in numerical_cols:
    df[col] = df[col].fillna(
        df[col].median()
    )

In [11]:
categorical_cols = [
    "City",
    "CountyOrParish"
]

for col in categorical_cols:
    df[col] = df[col].fillna("Unknown")

In [12]:
df.isnull().sum().sort_values(
    ascending=False
).head(20)

LotSizeSquareFeet        0
LivingArea               0
City                     0
BathroomsTotalInteger    0
ClosePrice               0
BedroomsTotal            0
CountyOrParish           0
ListPrice                0
CloseDate                0
PropertyAge              0
dtype: int64

In [13]:
df = pd.get_dummies(
    df,
    columns=[
        "City",
        "CountyOrParish"
    ],
    drop_first=True
)

In [14]:
test_df = df[
    df["CloseDate"] >= "2026-05-01"
]

train_df = df[
    df["CloseDate"] < "2026-05-01"
]

print(train_df.shape)
print(test_df.shape)

(49703, 976)
(12024, 976)


In [15]:
train_df = train_df.drop(
    columns=["CloseDate"]
)

test_df = test_df.drop(
    columns=["CloseDate"]
)

In [16]:
X_train = train_df.drop(
    columns=["ClosePrice"]
)

y_train = train_df["ClosePrice"]

X_test = test_df.drop(
    columns=["ClosePrice"]
)

y_test = test_df["ClosePrice"]

In [17]:
print(X_train.shape)
print(y_train.shape)

print(X_test.shape)
print(y_test.shape)

(49703, 974)
(49703,)
(12024, 974)
(12024,)


In [18]:
train_df.to_csv(
    "../data/train_processed.csv",
    index=False
)

test_df.to_csv(
    "../data/test_processed.csv",
    index=False
)

print("Saved successfully")

Saved successfully


# Week 3 Summary

Completed preprocessing pipeline including:

- Combined six months of California sales data
- Filtered to residential single-family homes
- Selected core predictive features
- Handled missing values
- Engineered PropertyAge feature
- Encoded categorical variables
- Created time-based train/test split
- Saved processed datasets for model development

The dataset is now ready for Week 4 baseline model development using Linear Regression.